In [ ]:
# Cell 0 — GPU check (must be T4 x2 for blip2-flan-t5-xl)

import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU count:     {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} — {mem:.1f} GB")

if torch.cuda.device_count() < 2:
    print("\n⚠️  Need GPU T4 x2 — go to Settings → Accelerator → GPU T4 x2")
else:
    print("\n✅ GPU T4 x2 confirmed — ready to run flan-t5-xl")

In [ ]:
# Cell 1 — Install & pull latest code

import os

!pip install transformers accelerate -q

REPO_DIR = "/kaggle/working/vision-search-engine"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/3-pi-radians/vision-search-engine.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

# Verify key config changes landed
print("\n=== Config verification ===")
!grep -n "BLIP2_MODEL_NAME\|CAPTION_BATCH_SIZE\|BLIP2_NUM_BEAMS\|BLIP2_MAX_NEW_TOKENS\|BLIP2_RERANK_MODEL_NAME" config.py

In [ ]:
# Cell 2 — Verify all inputs are attached

import os

checks = {
    "image_paths.json": "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/image_paths.json",
    "crops/gallery":    "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/crops/gallery",
    "crops/train":      "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-crops/crops/train",
    "annotations":      "/kaggle/input/datasets/pankajdeopa/deepfashion-inshop-annotations/list_eval_partition.txt",
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    if exists and os.path.isdir(path):
        count = sum(len(f) for _, _, f in os.walk(path))
        print(f"  {name}: {'✅' if exists else '❌'} ({count:,} files)")
    else:
        print(f"  {name}: {'✅' if exists else '❌ NOT FOUND'}")
    if not exists:
        all_ok = False

print("\n✅ Ready to run!" if all_ok else "\n❌ Fix missing inputs before running")

In [ ]:
# Cell 3 — Verify config paths and blip2_prompts.py exists

!grep -n "IMAGE_PATHS_PATH\|CROPS_DIR\|CAPTIONS_PATH\|BLIP2_MODEL_NAME\|CAPTION_BATCH_SIZE" config.py

print("\n=== blip2_prompts.py ===")
import os
prompts_path = "/kaggle/working/vision-search-engine/offline/blip2_prompts.py"
if os.path.exists(prompts_path):
    print("✅ blip2_prompts.py exists")
    !wc -l {prompts_path}
else:
    print("❌ blip2_prompts.py NOT FOUND — check git pull succeeded")

In [ ]:
# Cell 4 — Preview category prompts (sanity check before running)

import sys
sys.path.insert(0, "/kaggle/working/vision-search-engine")
from offline.blip2_prompts import CATEGORY_PROMPTS, DEFAULT_PROMPT

# Test a few categories
test_categories = ["Dresses", "Blouses_Shirts", "Denim", "Graphic_Tees", "unknown_category"]

for cat in test_categories:
    prompt = CATEGORY_PROMPTS.get(cat, DEFAULT_PROMPT)
    print(f"\n[{cat}]")
    print(f"  {prompt[:120]}..." if len(prompt) > 120 else f"  {prompt}")

In [ ]:
# Cell 5 — Run BLIP-2 captioning
# Expected time: ~45-60 min on GPU T4 x2
# Checkpoints every 500 items — resumable if session times out

!python offline/run_blip2_caption.py

In [ ]:
# Cell 6 — Verify output quality

import os, json

captions_path = "/kaggle/working/captions.json"

if not os.path.exists(captions_path):
    print("❌ captions.json not found — did Cell 5 complete?")
else:
    with open(captions_path) as f:
        captions = json.load(f)

    print(f"✅ captions.json found — {len(captions):,} captions")
    print(f"   Expected: 3,985")

    # Format check
    structured = sum(1 for c in captions.values() if ":" in c)
    short      = sum(1 for c in captions.values() if len(c.split()) < 6)
    has_person = sum(1 for c in captions.values() if
                     any(p in c.lower() for p in ["a woman", "a man", "the woman", "the man"]))

    print(f"\n=== Caption quality ===")
    print(f"  Structured (key:value format): {structured:,} ({100*structured/len(captions):.1f}%)")
    print(f"  Short (<6 words):              {short:,}")
    print(f"  Contains person reference:     {has_person:,}")

    # Sample 5 captions
    import random
    sample = random.sample(list(captions.items()), min(5, len(captions)))
    print(f"\n=== Sample captions ===")
    for item_id, caption in sample:
        print(f"\n  {item_id}:")
        print(f"  {caption}")

In [ ]:
# Cell 7 — Publish as new version of deepfashion-inshop-captions

import json, os

USERNAME = "pankajdeopa"

# Create clean publish folder
os.makedirs("/kaggle/working/captions-dataset", exist_ok=True)
!cp /kaggle/working/captions.json /kaggle/working/captions-dataset/captions.json

# Write metadata
meta = {
    "title": "deepfashion-inshop-captions",
    "id": f"{USERNAME}/deepfashion-inshop-captions",
    "licenses": [{"name": "CC0-1.0"}]
}
with open("/kaggle/working/captions-dataset/dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Publishing new version...")

# VERSION — not create (dataset already exists)
!kaggle datasets version \
    -p /kaggle/working/captions-dataset/ \
    -m "v2: blip2-flan-t5-xl, structured prompts 4+5+6, category-aware, front-view, person-ref stripped" \
    --dir-mode zip

print(f"\n✅ Published: https://www.kaggle.com/datasets/{USERNAME}/deepfashion-inshop-captions")

In [ ]:
# Cell 8 — Final summary

import os, json

print("=== BLIP-2 Captioning Complete ===")
print()
print("Model:    Salesforce/blip2-flan-t5-xl")
print("Strategy: 4+5+6 (fashion-specific + structured + category-aware)")
print("View:     front view preferred (01_1_front)")
print("Post:     person references stripped")
print()

captions_path = "/kaggle/working/captions.json"
if os.path.exists(captions_path):
    with open(captions_path) as f:
        captions = json.load(f)
    print(f"Total captions: {len(captions):,}")
    structured = sum(1 for c in captions.values() if ":" in c)
    print(f"Structured:     {structured:,} ({100*structured/len(captions):.1f}%)")

print()
print("=== Next steps ===")
print("1. ✅ captions published as deepfashion-inshop-captions v2")
print("2. ⏳ Run run_build_index.py in a new notebook")
print("   Attach: deepfashion-inshop-crops v2")
print("   Attach: deepfashion-inshop-captions v2  ← new")
print("   Attach: deepfashion-clip-weights v2")
print("3. ⏳ Publish updated hnsw indexes")
print("4. ⏳ Download locally and restart FastAPI")